# 09 Document-level Llama baseline

This notebook runs a zero-shot document-level simplification baseline with Ollama using llama3.1:8b.

It uses the document-level Cochrane-auto data, keeps the model frozen, and saves predictions to results/llama_document_level_predictions.csv.

In [ ]:
from __future__ import annotations

import gc
import json
import os
import random
import sys
import time
import urllib.error
import urllib.request
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from tqdm import tqdm

try:
    import evaluate
except ImportError as exc:
    raise ImportError("Install evaluation dependencies with: pip install evaluate bert-score") from exc

print("Imported core libraries for document-level Llama baseline.")

In [ ]:

def load_env_file(path: Path) -> None:
    """Load simple KEY=VALUE entries from a local .env file if present."""
    if not path.exists():
        return

    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ.setdefault(key, value)


PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_env_file(PROJECT_ROOT / ".env")

SEED = 42
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.1:8b")
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://127.0.0.1:11434")
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PREDICTION_PATH = RESULTS_DIR / "llama_document_level_predictions.csv"

GENERATION_CONFIG = {
    "max_new_tokens": 512,
    "temperature": 0.2,
    "top_p": 0.9,
    "seed": SEED,
}

OLLAMA_OPTIONS = {
    "num_predict": GENERATION_CONFIG["max_new_tokens"],
    "temperature": GENERATION_CONFIG["temperature"],
    "top_p": GENERATION_CONFIG["top_p"],
    "seed": SEED,
}


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)


set_seed(SEED)
print("Project root detected and generation settings configured.")

In [ ]:
DATA_DIR = PROJECT_ROOT / "data" / "document"
TRAIN_PATH = DATA_DIR / "cochraneauto_docs_train.csv"
VAL_PATH = DATA_DIR / "cochraneauto_docs_val.csv"
TEST_PATH = DATA_DIR / "cochraneauto_docs_test.csv"

for path in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    print(path.name, path.exists())

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))
print("Missing values:")
print(train_df.isna().sum())
print(val_df.isna().sum())
print(test_df.isna().sum())

In [ ]:
print("Document length statistics on the test split")

def word_count(text: str) -> int:
    return len(str(text).split())

for split_name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    lengths = df["complex"].fillna("").astype(str).map(word_count)
    print(split_name, "complex words", lengths.describe().to_dict())

print("Sample complex document:")
print(test_df.loc[0, "complex"][:1200])
print("\nSample reference simplification:")
print(test_df.loc[0, "simple"][:1200])

## Prompt template

The prompt follows the requested zero-shot style and asks the model to simplify the full biomedical document for a general audience.

In [ ]:
PROMPT_TEMPLATE = """You are an expert in biomedical text simplification.

Rewrite the following biomedical document for a general audience.

Rules:
- Preserve the main meaning and key findings.
- Use clear and simple language.
- Replace medical, scientific, or technical terms with simpler alternatives whenever possible.
- Remove unnecessary statistical or methodological details unless they are essential.
- Do not add information that is not present in the original document.
- Keep the output coherent and easy to read.

Document:
{complex_document}

Simplified document:
"""


def build_prompt(complex_document: str) -> str:
    return PROMPT_TEMPLATE.format(complex_document=str(complex_document).strip())

print("Built prompt template for document-level simplification.")

In [ ]:
def ollama_request(path: str, payload: dict[str, Any] | None = None, timeout: int = 120) -> dict[str, Any]:
    """Call the Ollama HTTP API with robust error handling."""
    url = f"{OLLAMA_URL}{path}"
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(url, data=data, headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.URLError as exc:
        message_lines = [
            f"Could not reach Ollama at {OLLAMA_URL}.",
            "Start Ollama in the same environment as this notebook, then rerun this cell.",
            "",
            "Local Mac command:",
            f"  ollama serve",
            f"  ollama pull {OLLAMA_MODEL}",
            "",
            "Quick checks:",
            f"  curl {OLLAMA_URL}/api/tags",
            "  ollama list",
        ]
        raise RuntimeError("\n".join(message_lines)) from exc


def list_ollama_models() -> list[str]:
    response = ollama_request("/api/tags", timeout=15)
    return [model["name"] for model in response.get("models", [])]


def model_is_available(model_name: str, available_models: list[str]) -> bool:
    if model_name in available_models:
        return True
    return any(name.startswith(f"{model_name}-") for name in available_models)


def ensure_ollama_model(model_name: str) -> None:
    available_models = list_ollama_models()
    if not model_is_available(model_name, available_models):
        available = ", ".join(available_models) if available_models else "no local models"
        raise RuntimeError(
            f"Ollama model {model_name} is not installed. Available: {available}. Install it with: ollama pull {model_name}"
        )
    print(f"Using Ollama model: {model_name}")


ensure_ollama_model(OLLAMA_MODEL)

In [ ]:
def clean_prediction(text: str) -> str:
    text = text.strip()
    prefixes = [
        "Simplified document:",
        "Simplified:",
        "Answer:",
        "Rewrite the following biomedical document for a general audience.",
    ]
    for prefix in prefixes:
        if text.lower().startswith(prefix.lower()):
            text = text[len(prefix):].strip()
    return " ".join(text.split())


def generate_prediction(complex_document: str) -> str:
    prompt = build_prompt(complex_document)
    response = ollama_request(
        "/api/generate",
        payload={
            "model": OLLAMA_MODEL,
            "prompt": prompt,
            "stream": False,
            "options": OLLAMA_OPTIONS,
        },
        timeout=300,
    )
    prediction = clean_prediction(response.get("response", ""))
    return prediction if prediction else "[EMPTY_OUTPUT]"

In [ ]:

def generate_predictions(input_df: pd.DataFrame) -> pd.DataFrame:
    """Generate predictions for document-level inputs with resume support and intermediate saving."""
    results = input_df.copy()
    results["prediction"] = ""

    if PREDICTION_PATH.exists():
        existing_df = pd.read_csv(PREDICTION_PATH)
        if "pair_id" in existing_df.columns and "prediction" in existing_df.columns:
            existing_map = existing_df.set_index("pair_id")["prediction"].to_dict()
            results["prediction"] = results["pair_id"].astype(str).map(existing_map).fillna("")

    pending_rows = results[results["prediction"].eq("")].copy()
    print("Rows already present:", len(results) - len(pending_rows))
    print("Rows to generate:", len(pending_rows))

    for idx, row in tqdm(pending_rows.iterrows(), total=len(pending_rows), desc="Generating"):
        try:
            prediction = generate_prediction(row["complex"])
        except Exception as exc:
            prediction = "[GENERATION_FAILED]"
            print(f"Generation failed at row {idx}: {exc}")

        results.loc[row.name, "prediction"] = prediction

        if idx % 10 == 0:
            results.to_csv(PREDICTION_PATH, index=False)
            print(f"Saved intermediate results at row {idx}")

    results.to_csv(PREDICTION_PATH, index=False)
    print(f"Saved final predictions to: {PREDICTION_PATH.relative_to(PROJECT_ROOT)}")
    return results

In [ ]:
# Use the test split for evaluation.
input_df = test_df[["pair_id", "complex", "simple"]].copy()
input_df["pair_id"] = input_df["pair_id"].astype(str)

prediction_df = generate_predictions(input_df)

print("First 3 generated examples")
print(prediction_df.head(3).to_string(index=False))

In [ ]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation import compute_metrics

metrics_summary = compute_metrics(prediction_df)
print(metrics_summary)

prediction_df["source_length"] = prediction_df["complex"].fillna("").astype(str).map(word_count)
prediction_df["prediction_length"] = prediction_df["prediction"].fillna("").astype(str).map(word_count)
prediction_df["reference_length"] = prediction_df["simple"].fillna("").astype(str).map(word_count)

avg_source_length = prediction_df["source_length"].mean()
avg_prediction_length = prediction_df["prediction_length"].mean()
avg_reference_length = prediction_df["reference_length"].mean()
compression_ratio = avg_prediction_length / avg_source_length if avg_source_length else float("nan")
empty_prediction_count = int((prediction_df["prediction"].fillna("").astype(str).str.strip() == "").sum())

print("Average source length:", round(avg_source_length, 2))
print("Average prediction length:", round(avg_prediction_length, 2))
print("Average reference length:", round(avg_reference_length, 2))
print("Compression ratio:", round(compression_ratio, 3))
print("Empty prediction count:", empty_prediction_count)